# Advanced Analysis: IV Gaussian Filter Data (b_scans)

This notebook performs comprehensive analysis on the Gaussian-filtered IV H-scan data.

**Note:** Temperature is extracted from the DataFrame as the single source of truth.

## Analysis Components:
1. I(H) curves at fixed voltages
2. IV characteristics at different magnetic fields
3. TMR ratio vs voltage
4. Quality metrics

## 1. Setup & Data Loading

In [ ]:
# Notebook setup
from scripts.utils import setup_notebook
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()

# Additional imports
from scipy.interpolate import griddata
from scripts.IV_Hscan_gaussian import load_dataframe, get_current_at_voltage, get_asymmetric_current_at_voltage

In [ ]:
# Load DataFrame - extract temperature from filename for simplicity
import glob
from pathlib import Path

# First, get the temperature from the notebook filename
# This notebook is named analysis_{temperature}K.ipynb
import re
notebook_name = globals().get('__vsc_ipynb_file__', '')
if not notebook_name:
    # Fallback: try to detect from current directory
    import os
    notebook_name = os.getcwd()

# Extract temperature from path or use a reasonable guess
temp_match = re.search(r'(\d+)K', notebook_name)
if temp_match:
    temperature = int(temp_match.group(1))
else:
    # If we can't detect, we'll extract from DataFrame after loading
    temperature = None

# Try to load dataframe - prefer simple name without timestamp
df_path = PROJECT_ROOT / r"output/IV_H_scans/dataframes/b_scans"

if temperature is not None:
    # Try simple pattern first (IV_gaussian_{T}K.pkl)
    simple_path = df_path / f"IV_gaussian_{temperature}K.pkl"
    if simple_path.exists():
        df_path = simple_path
    else:
        # Fallback: try pattern with timestamp
        pattern = str(df_path / f"IV_gaussian_{temperature}K*.pkl")
        matching_files = glob.glob(pattern)
        if matching_files:
            # Prefer files without extra timestamp suffix
            simple_files = [f for f in matching_files if re.match(r'.*_\d+K\.pkl$', f)]
            df_path = Path(simple_files[0] if simple_files else matching_files[0])
        else:
            raise FileNotFoundError(f"No dataframe found for {temperature}K in b_scans")
else:
    # Load any file if temperature detection failed
    all_files = list(df_path.glob('IV_gaussian_*.pkl'))
    if all_files:
        df_path = all_files[0]
    else:
        raise FileNotFoundError(f"No dataframe files found in b_scans")

df = load_dataframe(df_path)

# Extract temperature from DataFrame as SINGLE SOURCE OF TRUTH
if 'temperature' in df.columns:
    temperature = int(df['temperature'].iloc[0])
else:
    raise ValueError("Temperature column not found in DataFrame!")

print(f"\n{'='*60}")
print(f"DataFrame loaded successfully!")
print(f"Loaded from: {df_path.name}")
print(f"Temperature (from DataFrame): {temperature} K")
print(f"{'='*60}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"H field range: {df['H'].min():.4f} to {df['H'].max():.4f} T")
print(f"Number of measurements: {len(df)}")

## 2. I(H) Curves at Fixed Voltages

Plot current vs magnetic field at specific bias voltages.

In [ ]:
import matplotlib.cm as cm
from scripts.utils import OKABE_ITO_CYCLE
from scripts.utils import OKABE_ITO

voltages_to_plot = [0.5]
print(f"Will plot I(H) curves at {len(voltages_to_plot)} voltages.")


fig, ax = plt.subplots(figsize=(6, 5), dpi=600)

for V_val in voltages_to_plot:
    I_vs_H = get_current_at_voltage(df, V_val, use_filtered=True)
    ax.plot(I_vs_H['H'], 1E9*I_vs_H['I_at_V'], 'o-', color=OKABE_ITO_CYCLE[1], markersize=4, linewidth=2, alpha=0.8)



ax.set_xlabel('$H_Y$ (T)', fontsize=14)
ax.set_ylabel('$I$ (nA)', fontsize=14)
#ax.set_title(f'I(H) Curves at Fixed Voltages - {temperature}K', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.cm as cm

voltages_to_plot = np.linspace(-1, 1, 51)
print(f"Will plot I(H) curves at {len(voltages_to_plot)} voltages.")

cmap = cm.get_cmap('RdBu_r')
norm = plt.Normalize(vmin=voltages_to_plot.min(), vmax=voltages_to_plot.max())

fig, ax = plt.subplots(figsize=(12, 7))

for V_val in voltages_to_plot:
    I_vs_H = get_current_at_voltage(df, V_val, use_filtered=True)
    color = cmap(norm(V_val))
    ax.plot(I_vs_H['H'], I_vs_H['I_at_V'] , 'o-', color=color, markersize=4, linewidth=2, alpha=0.8)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('Voltage (V)', fontsize=14)

ax.set_xlabel('Magnetic Field H (T)', fontsize=14)
ax.set_ylabel('Current (A)', fontsize=14)
ax.set_title(f'I(H) Curves at Fixed Voltages - {temperature}K', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Build 2D arrays for the colormap ---
from matplotlib.pyplot import tick_params
df_sorted = df.sort_values('H').reset_index(drop=True)

V_grid = np.vstack(df_sorted['voltage_smooth'].values)   # (n_H, 500)
I_grid = 1E9*np.vstack(df_sorted['current_smooth'].values)   # (n_H, 500)
H_vals = df_sorted['H'].values                           # (n_H,)

# --- H values at which to overlay IV curves ---
H_targets = [-0.8, -0.5, -0.2, 0.0, 0.2, 0.5, 0.8]   # T

cmap_iv = plt.cm.viridis
h_norm  = plt.Normalize(vmin=H_vals.min(), vmax=H_vals.max())

fig, ax = plt.subplots(figsize=(6, 5), dpi=600)



# --- Corrected plotting block ---
pcm = ax.pcolormesh(
    V_grid,                                          # x
    np.tile(H_vals[:, None], (1, V_grid.shape[1])),  # y
    I_grid,                                          # color
    cmap='RdBu_r',
    shading='auto'
)

# Set your custom ticks here (e.g., from -15 to 15 nA)
cbar_map = fig.colorbar(pcm, ax=ax, ticks=[-400, -300, -200, -100, 0, 100, 200, 300, 400])
cbar_map.set_label('$I$ (nA)')


ax.set_xlabel('$V_{bias}$ (V)')
ax.set_ylabel('$H_Y$ (T)')
#ax.set_title(f'I(V, H) Colormap — {temperature} K', fontsize=13)



plt.tight_layout()
plt.show()

## 3. IV Colormap & IV Curves at Selected Fields

## 3. TMR Ratio Calculation

In [ ]:


# --- Configuration ---
voltage_points = np.linspace(-1.0, 1.0, 200)
H_low_threshold = 0.1   # T - low field region (for anti-parallel state)
H_high_threshold = 0.5  # T - high field region (for parallel state)

# --- Initialization ---
tmr_values = []
tmr_errors = []
i_apar_values = []
i_apar_errors = []
i_par_values = []
i_par_errors = []

# --- Main Calculation Loop ---
for V_target in voltage_points:
    data = get_current_at_voltage(df, V_target, use_filtered=True)
    H_vals = data['H'].values
    I_vals = data['I_at_V'].values

    # --- Calculate Anti-Parallel Current (I_apar) ---
    # This corresponds to the low-field region (previously I_min)
    low_field_mask = np.abs(H_vals) < H_low_threshold
    if low_field_mask.sum() > 1: # Need at least 2 points for std deviation
        currents_at_low_field = I_vals[low_field_mask]
        I_apar = np.mean(currents_at_low_field)
        # Use ddof=1 for sample standard deviation
        I_apar_err = np.std(currents_at_low_field, ddof=1) / np.sqrt(len(currents_at_low_field))
    else:
        I_apar, I_apar_err = np.nan, np.nan

    # --- Calculate Parallel Current (I_par) ---
    # This corresponds to the high-field region (previously I_max)
    high_field_mask = np.abs(H_vals) > H_high_threshold
    if high_field_mask.sum() > 1: # Need at least 2 points for std deviation
        currents_at_high_field = I_vals[high_field_mask]
        I_par = np.mean(currents_at_high_field)
        # Use ddof=1 for sample standard deviation
        I_par_err = np.std(currents_at_high_field, ddof=1) / np.sqrt(len(currents_at_high_field))
    else:
        I_par, I_par_err = np.nan, np.nan

    # --- Store Current Values ---
    i_apar_values.append(I_apar)
    i_apar_errors.append(I_apar_err)
    i_par_values.append(I_par)
    i_par_errors.append(I_par_err)

    # --- Calculate and Store TMR ---
    # Check for valid numbers and non-trivial current before division
    if not np.isnan(I_par) and not np.isnan(I_apar):
        tmr = I_par / I_apar
        # Propagate error for the division
        tmr_error = abs(tmr) * np.sqrt((I_par_err / I_par)**2 + (I_apar_err / I_apar)**2)
        tmr_values.append(abs(tmr))
        tmr_errors.append(tmr_error)
    else:
        tmr_values.append(np.nan)
        tmr_errors.append(np.nan)

# --- Finalize and Print Results ---
# Convert lists to NumPy arrays for easier analysis
tmr_values = np.array(tmr_values)
tmr_errors = np.array(tmr_errors)
i_apar_values = np.array(i_apar_values)
i_apar_errors = np.array(i_apar_errors)
i_par_values = np.array(i_par_values)
i_par_errors = np.array(i_par_errors)

print("TMR calculation complete!")
print(f"Voltage range: {voltage_points.min():.2f} to {voltage_points.max():.2f} V")
# Use nanmin/nanmax to safely get min/max in the presence of NaN values
print(f"TMR ratio range: {np.nanmin(tmr_values):.3f} to {np.nanmax(tmr_values):.3f}")



## 4. Plot TMR Ratio vs Voltage

In [ ]:
plt.figure(figsize=(12, 7))

plt.plot(voltage_points, tmr_values, '-', color='dodgerblue', linewidth=2, label='TMR Ratio')
plt.fill_between(voltage_points, tmr_values - tmr_errors, tmr_values + tmr_errors, color='dodgerblue', alpha=0.2, label='Propagated Error')

plt.xlabel("Voltage (V)", fontsize=14)
plt.ylabel("TMR Ratio (I_max / I_min)", fontsize=14)
plt.title(f"TMR Ratio vs. Applied Voltage - {temperature}K", fontsize=16)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 5. Save TMR Data

In [ ]:
data_to_save = {
    'Voltage (V)': voltage_points,
    'TMR_Ratio': tmr_values,
    'TMR_Error': tmr_errors,
    'I_apar (A)': i_apar_values,
    'I_apar_error (A)': i_apar_errors,
    'I_par (A)': i_par_values,
    'I_par_error (A)': i_par_errors
}

df_results = pd.DataFrame(data_to_save)
output_path = PROJECT_ROOT / r"output/IV_H_scans/MR_summary/b_scans"
output_path.mkdir(parents=True, exist_ok=True)
filename = f"TMR_ratio_vs_V_{temperature}K.csv"
df_results.to_csv(output_path / filename, index=False)

print(f"Data successfully saved to: {filename}")
print("\nFirst 5 rows of the data saved:")
print(df_results.head())